# 2. Monte Carlo Dropout

Train a regression network with dropout and draw stochastic predictions through
`MCDropoutEnsembler`. See [setup and running instructions](README.md).
All computation uses CPU and synthetic data.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn

from bensemble.diversity import MCDropoutEnsembler

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small CPU workloads avoid thread-pool overhead.
DEVICE = torch.device("cpu")
EPOCHS = 600
NUM_SAMPLES = 50

## 1. Data and evaluation protocol

Generate a noisy regression problem and split it into disjoint train, validation,
and test sets. Training inputs lie in `[-3, 3]`; the plotting grid extends to
`[-5, 5]`. Test RMSE measures interpolation on fresh noisy targets. The wider grid
illustrates extrapolation, without assuming that uncertainty must grow outside
the training range.

The validation split is unused in this fixed-configuration demo; it is available for tuning dropout or training duration.

In [ ]:
def true_function(x):
    return torch.sin(1.5 * x) + 0.2 * x


generator = torch.Generator().manual_seed(SEED)
features = 6 * torch.rand(300, 1, generator=generator) - 3
targets = true_function(features) + 0.15 * torch.randn(300, 1, generator=generator)
indices = torch.randperm(len(features), generator=generator)
split_indices = {
    "train": indices[:180],
    "validation": indices[180:240],
    "test": indices[240:],
}
x_train, y_train = features[split_indices["train"]], targets[split_indices["train"]]
x_val, y_val = (
    features[split_indices["validation"]],
    targets[split_indices["validation"]],
)
x_test, y_test = features[split_indices["test"]], targets[split_indices["test"]]
x_grid = torch.linspace(-5, 5, 240).reshape(-1, 1)

plt.figure(figsize=(8, 4))
plt.scatter(x_train[:, 0], y_train[:, 0], s=12, label="Train")
plt.plot(x_grid[:, 0], true_function(x_grid)[:, 0], "k--", label="True function")
plt.axvspan(-3, 3, alpha=0.08, color="green", label="Training input range")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## 2. Train with dropout

Dropout is active during training. At prediction time the adapter activates
dropout while keeping other layers in evaluation mode.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(), nn.Dropout(p=0.1), nn.Linear(64, 1)
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model.train()
for _ in range(EPOCHS):
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(model(x_train), y_train)
    loss.backward()
    optimizer.step()

model.eval()
ensemble = MCDropoutEnsembler(model).build_ensemble(num_samples=NUM_SAMPLES)

## 3. Sample predictions and evaluate

`predict_members` has shape `(samples, batch, outputs)`. The sample standard
deviation measures variation between dropout predictions. The band below excludes
observation noise and is **not a calibrated 95% prediction interval**.

In [ ]:
with torch.no_grad():
    member_predictions = ensemble.predict_members(x_grid)
    mean = member_predictions.mean(dim=0)
    std = member_predictions.std(dim=0)
    test_mean = ensemble.predict_members(x_test).mean(dim=0)

assert member_predictions.shape == (NUM_SAMPLES, len(x_grid), 1)
assert torch.isfinite(member_predictions).all()
assert std.max() > 0, "Dropout sampling should produce different predictions"
test_rmse = (test_mean - y_test).square().mean().sqrt().item()
print(f"Held-out test RMSE: {test_rmse:.3f}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.scatter(x_train[:, 0], y_train[:, 0], s=12, alpha=0.5, label="Train")
plt.plot(x_grid[:, 0], true_function(x_grid)[:, 0], "k--", label="True function")
plt.plot(x_grid[:, 0], mean[:, 0], label="Predictive mean")
plt.fill_between(
    x_grid[:, 0],
    (mean - 2 * std)[:, 0],
    (mean + 2 * std)[:, 0],
    alpha=0.2,
    label="Mean ± 2 model standard deviations",
)
plt.axvline(-3, color="grey", linestyle=":")
plt.axvline(3, color="grey", linestyle=":")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

Next: [Variational inference](03_variational_inference.ipynb) learns a posterior and an observation-noise scale.